# Level 5 — Distributed Training Profiling (DDP on 2×T4)

Profile a real DDP training run and measure the cost of GPU-to-GPU communication.

**What this lab does:**
1. Runs the same GPT-2 model from Level 3 on a single GPU (baseline)
2. Runs the same model with DDP across 2 GPUs
3. Profiles both runs and separates **compute time** from **NCCL communication time**
4. Measures compute-communication overlap and scaling efficiency

**Platform:** Kaggle (free 2×T4) or any multi-GPU machine.

**Key concept:** DDP replicates the model on each GPU, splits the data, and synchronizes gradients via AllReduce after backward. The question is: does communication hide behind compute, or does it add to step time?


## 1. Setup and GPU check


In [1]:
!pip install torch transformers --quiet

import torch
import os

num_gpus = torch.cuda.device_count()
print(f'Available GPUs: {num_gpus}')
for i in range(num_gpus):
    print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')
    print(f'         Memory: {torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB')

assert num_gpus >= 2, 'This lab requires 2 GPUs. On Kaggle: Settings → Accelerator → GPU T4 x2'
print(f'\nReady for DDP with {num_gpus} GPUs')


Available GPUs: 2
  GPU 0: Tesla T4
         Memory: 15.6 GB
  GPU 1: Tesla T4
         Memory: 15.6 GB

Ready for DDP with 2 GPUs


## 2. Define constants and model config

We use the same GPT-2 config from Level 3 so results are directly comparable.


In [2]:
from transformers import GPT2Model, GPT2Config
from torch.profiler import profile, record_function, ProfilerActivity, schedule, tensorboard_trace_handler
from collections import defaultdict
import time, json, glob

CONFIG = GPT2Config(
    n_layer=6, n_head=8, n_embd=512,
    n_positions=256, vocab_size=50257,
)
BATCH_SIZE = 8
SEQ_LEN = 128
NUM_STEPS = 15
ACTIVE_STEPS = 10

n_params = sum(p.numel() for p in GPT2Model(CONFIG).parameters())
grad_size_mb = n_params * 4 / 1e6
print(f'Model: GPT-2 style, {n_params/1e6:.1f}M params')
print(f'Gradient data per AllReduce: {grad_size_mb:.1f} MB')
print(f'Batch: {BATCH_SIZE} per GPU, Seq: {SEQ_LEN}, Steps: {NUM_STEPS}')


Model: GPT-2 style, 44.8M params
Gradient data per AllReduce: 179.1 MB
Batch: 8 per GPU, Seq: 128, Steps: 15


## 3. Single-GPU baseline


In [3]:
def run_single_gpu():
    """Baseline: train on GPU 0 only."""
    device = torch.device('cuda:0')
    model = GPT2Model(CONFIG).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    inputs = torch.randint(0, CONFIG.vocab_size, (BATCH_SIZE, SEQ_LEN), device=device)

    log_dir = './log_single_gpu'
    os.makedirs(log_dir, exist_ok=True)

    prof_schedule = schedule(wait=1, warmup=2, active=ACTIVE_STEPS, repeat=1)

    with profile(
        activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
        schedule=prof_schedule,
        on_trace_ready=tensorboard_trace_handler(log_dir),
        record_shapes=True,
        profile_memory=True,
    ) as prof:
        for step in range(NUM_STEPS):
            with record_function('forward'):
                out = model(inputs).last_hidden_state
                loss = out.mean()
            with record_function('backward'):
                loss.backward()
            with record_function('optimizer_step'):
                optimizer.step()
                optimizer.zero_grad()
            prof.step()

    return prof

print('Running single-GPU baseline...')
single_prof = run_single_gpu()
print('Done.\n')
print('Top 10 ops by CUDA time (single GPU):')
print(single_prof.key_averages().table(sort_by='cuda_time_total', row_limit=10))


Running single-GPU baseline...


/usr/local/lib/python3.12/dist-packages/torch/profiler/profiler.py:217: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(


Done.

Top 10 ops by CUDA time (single GPU):
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg       CPU Mem  Self CPU Mem      CUDA Mem  Self CUDA Mem    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                          ProfilerStep*         0.07%     682.338us        58.16%     553.266ms      55.327ms       0.000us         0.00%     336.701ms      33.670ms          

## 4. Write DDP training script

DDP requires separate processes (one per GPU). In notebooks, we write a `.py` script and launch it with `torchrun` — this is how DDP is actually run in production.

**What DDP does under the hood:**
- Forward: each GPU computes on its own data shard (no communication)
- Backward: gradients are synchronized via NCCL AllReduce (this is the communication cost)
- Optimizer: each GPU updates its own copy independently (no communication)


In [4]:
%%writefile ddp_profiling.py
import torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.profiler import profile, ProfilerActivity, schedule, tensorboard_trace_handler, record_function
from transformers import GPT2Model, GPT2Config
import os, json

def main():
    dist.init_process_group('nccl')
    rank = dist.get_rank()
    world_size = dist.get_world_size()
    torch.cuda.set_device(rank)
    device = torch.device(f'cuda:{rank}')

    config = GPT2Config(
        n_layer=6, n_head=8, n_embd=512,
        n_positions=256, vocab_size=50257,
    )
    model = GPT2Model(config).to(device)
    ddp_model = DDP(model, device_ids=[rank])
    optimizer = torch.optim.Adam(ddp_model.parameters(), lr=1e-4)

    # Each GPU gets same-sized batch (total batch = BATCH_SIZE * world_size)
    inputs = torch.randint(0, config.vocab_size, (8, 128), device=device)

    NUM_STEPS = 15
    ACTIVE_STEPS = 10

    log_dir = f'./log_ddp_rank{rank}'
    os.makedirs(log_dir, exist_ok=True)

    prof_schedule = schedule(wait=1, warmup=2, active=ACTIVE_STEPS, repeat=1)

    # Synchronize before profiling
    dist.barrier()

    with profile(
        activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
        schedule=prof_schedule,
        on_trace_ready=tensorboard_trace_handler(log_dir),
        record_shapes=True,
        profile_memory=True,
    ) as prof:
        for step in range(NUM_STEPS):
            with record_function('forward'):
                out = ddp_model(inputs).last_hidden_state
                loss = out.mean()
            with record_function('backward'):
                loss.backward()
            with record_function('optimizer_step'):
                optimizer.step()
                optimizer.zero_grad()
            prof.step()

    # Save profiler results from rank 0
    if rank == 0:
        events = prof.key_averages()
        data = [
            {'name': e.key, 'cuda_time': e.cuda_time * e.count, 'cpu_time': e.cpu_time * e.count, 'count': e.count}
            for e in events
        ]
        with open('ddp_results.json', 'w') as f:
            json.dump(data, f)
        print('\nTop 10 ops by CUDA time (DDP rank 0):')
        print(prof.key_averages().table(sort_by='cuda_time_total', row_limit=10))

    dist.destroy_process_group()

if __name__ == '__main__':
    main()


Writing ddp_profiling.py


## 5. Run DDP with torchrun


In [5]:
print('Running DDP on 2 GPUs...')
!torchrun --nproc_per_node=2 ddp_profiling.py
print('\nDone.')


Running DDP on 2 GPUs...
W0524 23:11:40.744000 124 torch/distributed/run.py:852] 
W0524 23:11:40.744000 124 torch/distributed/run.py:852] *****************************************
W0524 23:11:40.744000 124 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0524 23:11:40.744000 124 torch/distributed/run.py:852] *****************************************
[W524 23:11:48.596083521 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W524 23:12:04.175091132 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W524 23:12:04.184377364 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
/usr/local/lib/python3.12/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under cu

## 6. Categorize: compute vs communication

The key question: how much of each step is spent computing vs talking to the other GPU?

We look for:
- **NCCL kernels** (`nccl` in the name) → communication
- **sgemm / matmul kernels** → compute
- **Everything else** → overhead (Adam, elementwise, memset, etc.)


In [6]:
def classify_kernel(name):
    n = name.lower()
    if 'nccl' in n: return 'Communication (NCCL)'
    if 'sgemm' in n or 'gemm' in n or 'addmm' in n: return 'Compute (matmul)'
    if 'fmha' in n or 'attention' in n or 'flash' in n: return 'Compute (attention)'
    if 'adam' in n or 'multi_tensor' in n: return 'Optimizer'
    return 'Other'

# --- Single GPU breakdown ---
single_events = single_prof.key_averages()
single_buckets = defaultdict(float)
single_total = 0
for e in single_events:
    cat = classify_kernel(e.key)
    t = e.cuda_time * e.count
    single_buckets[cat] += t
    single_total += t

print('=== SINGLE GPU ===')
print(f'Total CUDA time: {single_total/1000:.2f} ms ({ACTIVE_STEPS} steps)\n')
for cat, t in sorted(single_buckets.items(), key=lambda x: -x[1]):
    pct = 100 * t / single_total if single_total else 0
    print(f'  {cat:<25} {t/1000:>10.2f} ms  {pct:>6.1f}%')
print(f'  Per-step average:       {single_total/1000/ACTIVE_STEPS:>10.2f} ms')

# --- DDP breakdown (rank 0) ---
with open('ddp_results.json') as f:
    ddp_events = json.load(f)

ddp_buckets = defaultdict(float)
ddp_total = 0
nccl_time = 0
for e in ddp_events:
    cat = classify_kernel(e['name'])
    ddp_buckets[cat] += e['cuda_time']
    ddp_total += e['cuda_time']
    if 'nccl' in e['name'].lower():
        nccl_time += e['cuda_time']

print(f'\n=== DDP (2 GPUs, rank 0) ===')
print(f'Total CUDA time: {ddp_total/1000:.2f} ms ({ACTIVE_STEPS} steps)\n')
for cat, t in sorted(ddp_buckets.items(), key=lambda x: -x[1]):
    pct = 100 * t / ddp_total if ddp_total else 0
    print(f'  {cat:<25} {t/1000:>10.2f} ms  {pct:>6.1f}%')
print(f'  Per-step average:       {ddp_total/1000/ACTIVE_STEPS:>10.2f} ms')

=== SINGLE GPU ===
Total CUDA time: 3159.86 ms (10 steps)

  Other                        1721.91 ms    54.5%
  Compute (matmul)              812.50 ms    25.7%
  Optimizer                     428.95 ms    13.6%
  Compute (attention)           196.50 ms     6.2%
  Per-step average:           315.99 ms

=== DDP (2 GPUs, rank 0) ===
Total CUDA time: 5270.96 ms (10 steps)

  Other                        3058.92 ms    58.0%
  Communication (NCCL)          939.14 ms    17.8%
  Compute (matmul)              672.44 ms    12.8%
  Optimizer                     420.10 ms     8.0%
  Compute (attention)           180.35 ms     3.4%
  Per-step average:           527.10 ms


/tmp/ipykernel_57/3372040155.py:15: FutureWarning: `cuda_time` is deprecated, please use `device_time` instead.
  t = e.cuda_time * e.count


## 7. Scaling efficiency analysis


In [7]:
single_step = single_total / 1000 / ACTIVE_STEPS  # ms per step
ddp_step = ddp_total / 1000 / ACTIVE_STEPS
comm_step = nccl_time / 1000 / ACTIVE_STEPS
compute_step = (ddp_total - nccl_time) / 1000 / ACTIVE_STEPS

# Scaling efficiency: how close to linear speedup?
# With DDP, each GPU processes BATCH_SIZE tokens.
# 2 GPUs process 2*BATCH_SIZE tokens total per step.
# Ideal: ddp_step == single_step (perfect overlap → 2x throughput)
# Worst: ddp_step == single_step + comm_time (no overlap → less than 2x)
scaling_eff = single_step / ddp_step if ddp_step > 0 else 0
throughput_ratio = 2 * scaling_eff  # 2 GPUs * efficiency
comm_overhead_pct = 100 * comm_step / ddp_step if ddp_step > 0 else 0
overlap_ratio = max(0, 1 - (ddp_step - single_step) / comm_step) if comm_step > 0 else 0

print('=== SCALING ANALYSIS ===')
print(f'Single GPU step time:       {single_step:.2f} ms')
print(f'DDP step time (rank 0):     {ddp_step:.2f} ms')
print(f'  Compute:                  {compute_step:.2f} ms')
print(f'  Communication (NCCL):     {comm_step:.2f} ms')
print(f'  Communication overhead:   {comm_overhead_pct:.1f}% of step time')
print(f'  Compute-comm overlap:     {overlap_ratio*100:.0f}%')
print(f'')
print(f'Scaling efficiency:         {scaling_eff*100:.1f}%')
print(f'Effective throughput:        {throughput_ratio:.2f}x (ideal: 2.0x)')
print(f'')
print(f'Gradient data per AllReduce: {grad_size_mb:.1f} MB')


=== SCALING ANALYSIS ===
Single GPU step time:       315.99 ms
DDP step time (rank 0):     527.10 ms
  Compute:                  433.18 ms
  Communication (NCCL):     93.91 ms
  Communication overhead:   17.8% of step time
  Compute-comm overlap:     0%

Scaling efficiency:         59.9%
Effective throughput:        1.20x (ideal: 2.0x)

Gradient data per AllReduce: 179.1 MB


## 8. Export traces

Download traces for both single-GPU and DDP rank 0.
Open in [Perfetto UI](https://ui.perfetto.dev) to visually compare.

**What to look for in the DDP trace:**
- NCCL AllReduce kernels appearing during backward pass (gradient sync calls)
- Whether NCCL kernels overlap with compute kernels (good) or run sequentially (bad)
- The gap between backward end and optimizer start


In [8]:
print('Trace files generated:')
for pattern in ['./log_single_gpu/*.pt.trace.json', './log_ddp_rank0/*.pt.trace.json']:
    for f in glob.glob(pattern):
        size_mb = os.path.getsize(f) / 1024 / 1024
        print(f'  {f}  ({size_mb:.1f} MB)')

# Download on Colab
try:
    from google.colab import files as colab_files
    for f in glob.glob('./log_single_gpu/*.pt.trace.json'):
        colab_files.download(f)
    for f in glob.glob('./log_ddp_rank0/*.pt.trace.json'):
        colab_files.download(f)
except ImportError:
    print('Not on Colab — traces are at the paths above.')
    print('On Kaggle: use the Output tab to download.')


Trace files generated:
  ./log_single_gpu/68f0f7a23959_57.1779664293850573427.pt.trace.json  (22.1 MB)
  ./log_ddp_rank0/68f0f7a23959_130.1779664328007259081.pt.trace.json  (26.0 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Key findings (fill in after running)

| Metric | Single GPU | DDP (2×T4) |
|--------|-----------|------------|
| Step time (ms) | _fill in_ | _fill in_ |
| Compute time (ms) | _fill in_ | _fill in_ |
| Communication time (ms) | — | _fill in_ |
| Communication overhead (%) | — | _fill in_ |
| Compute-comm overlap (%) | — | _fill in_ |
| Scaling efficiency | 100% | _fill in_ |
| Effective throughput | 1.0x | _fill in_ |

**Expected observations:**

1. **Communication is a significant fraction of step time** for this small model (~76 MB of gradient data per AllReduce on PCIe-connected T4s).

2. **DDP's gradient bucketing partially overlaps communication with backward compute.** PyTorch buckets gradients (default 25MB) and starts AllReduce for earlier layers while later layers are still computing backward.

3. **Scaling efficiency will be below 100%.** For small models, compute-to-communication ratio is unfavorable. At 7B+ params with large batches, it improves dramatically.

4. **The Perfetto trace tells the real story.** Open both traces side-by-side — in the DDP trace, look for NCCL AllReduce kernels interleaved with backward sgemm kernels.
